In [1]:
import os
import pandas as pd
import torch
from PIL import Image
# ⭐️⭐️⭐️ 이 부분이 수정되었습니다 ⭐️⭐️⭐️
from transformers import MllamaForConditionalGeneration, AutoProcessor
from tqdm import tqdm
import re

# --- 1. 경로 설정 ---
BASE_DATA_PATH = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/'
test_csv_path = os.path.join(BASE_DATA_PATH, 'dev_test.csv')
submission_csv_path = os.path.join(BASE_DATA_PATH, 'sample_submission.csv')
image_dir = os.path.join(BASE_DATA_PATH, 'input_images')
output_csv_path = os.path.join('./', 'submission.csv')

print("경로 설정 완료.")

# --- 2. 모델 및 프로세서 로드 ---
print("\n모델 및 프로세서 로드를 시작합니다...")
model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
try:
    # ⭐️⭐️⭐️ 이 부분이 수정되었습니다 ⭐️⭐️⭐️
    # 범용 클래스 대신 전용 클래스를 사용하여 모델을 로드합니다.
    model = MllamaForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    processor = AutoProcessor.from_pretrained(model_id)
    print("모델 및 프로세서 로드 완료.")
except Exception as e:
    print(f"모델 로드 중 오류 발생: {e}")
    exit()

# --- 3. 데이터 로드 ---
print("\nCSV 파일 로드를 시작합니다...")
test_df = pd.read_csv(test_csv_path)
submission_df = pd.read_csv(submission_csv_path)
print("CSV 파일 로드 완료.")

# --- 4. 답변 생성 및 제출 파일 업데이트 ---
print("\n이미지 분석 및 답변 생성을 시작합니다...")

for index, row in tqdm(test_df.iterrows(), total=test_df.shape[0], desc="답변 생성 중"):
    image_id = row['ID']
    image_filename = os.path.basename(row['img_path'])
    image_path = os.path.join(image_dir, image_filename)

    if not os.path.exists(image_path):
        print(f"경고: 이미지 파일을 찾을 수 없습니다. 건너<binary data, 2 bytes>니다: {image_path}")
        continue

    image = Image.open(image_path).convert("RGB")

    question = row['Question']
    options = f"A: {row['A']}\nB: {row['B']}\nC: {row['C']}\nD: {row['D']}"
    prompt_text = (
        f"Based on the image, answer the following multiple-choice question.\n\n"
        f"Question: {question}\n\n"
        f"Options:\n{options}\n\n"
        f"Provide only the letter (A, B, C, or D) of the most appropriate answer."
    )

    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_text}
        ]}
    ]
    
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    
    inputs = processor(
        text=input_text, 
        images=image, 
        return_tensors="pt"
    ).to(model.device)

    # 답변 생성
    generated_ids = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    
    # 결과 디코딩 및 파싱
    generated_text = processor.batch_decode(generated_ids[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]
    
    found_options = re.findall(r'\b([A-D])\b', generated_text)
    
    final_answer = '-'
    if found_options:
        final_answer = found_options[0]

    # submission DataFrame 업데이트
    submission_df.loc[submission_df['ID'] == image_id, 'answer'] = final_answer

print("답변 생성이 완료되었습니다.")

# --- 5. 결과 저장 ---
submission_df.to_csv(output_csv_path, index=False)
print(f"\n결과가 성공적으로 '{output_csv_path}' 파일에 저장되었습니다.")

print("\n생성된 제출 파일 (상위 5개):")
print(submission_df.head())

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


경로 설정 완료.

모델 및 프로세서 로드를 시작합니다...


Loading checkpoint shards: 100%|██████████| 5/5 [00:05<00:00,  1.08s/it]


모델 및 프로세서 로드 완료.

CSV 파일 로드를 시작합니다...
CSV 파일 로드 완료.

이미지 분석 및 답변 생성을 시작합니다...


답변 생성 중: 100%|██████████| 60/60 [00:31<00:00,  1.88it/s]

답변 생성이 완료되었습니다.

결과가 성공적으로 './submission.csv' 파일에 저장되었습니다.

생성된 제출 파일 (상위 5개):
         ID answer
0  TEST_000      A
1  TEST_001      B
2  TEST_002      C
3  TEST_003      A
4  TEST_004      B
